# PEDAP User Data Generation

This notebook generates user data expansion for the PEDAP dataset following the exact same pattern as DCLP3.csv structure.

## Dataset Overview
PEDAP (Pediatric Artificial Pancreas) Public Dataset contains data from a randomized controlled trial comparing:
- CLC (Control Loop Closed): Automated insulin delivery using Control-IQ technology
- SC (Standard Care): Standard pump therapy with Basal-IQ (predictive low glucose suspend)

The study was conducted in pediatric participants (ages 2-5 years) using Tandem t:slim X2 pumps with Dexcom G6 CGM.

In [1]:
import pandas as pd
import numpy as np
import os

## Load PEDAP Data Files

In [2]:
# Define data paths
base_path = '/Users/miriamk.wolff/Documents/Repositories/Replica/egvinsulin/data/raw/PEDAP Public Dataset - Release 3 - 2024-09-25/Data Files'

# Load roster data
roster = pd.read_csv(os.path.join(base_path, 'PtRoster.txt'), delimiter='|')
print(f"Roster shape: {roster.shape}")
print("Treatment groups:")
print(roster['TrtGroup'].value_counts())
roster.head()

Roster shape: (109, 8)
Treatment groups:
TrtGroup
CLC    68
SC     34
Name: count, dtype: int64


,RecID,PtID,SiteID,EnrollDt,RandDt,PtStatus,TrtGroup,AgeAsofEnrollDt
0,2,93,3,11/8/2020,11/20/2020 3:48:07 PM,Completed,CLC,2
1,4,26,3,11/9/2020,12/4/2020 4:06:49 PM,Completed,CLC,4
2,8,57,3,11/29/2020,12/11/2020 4:14:45 PM,Completed,SC,4
3,13,82,3,10/7/2020,10/14/2020 11:42:47 AM,Completed,CLC,2
4,14,44,3,12/13/2020,1/7/2021 1:19:26 PM,Completed,CLC,3


In [3]:
# Load screening data for device and demographic information
screening = pd.read_csv(os.path.join(base_path, 'PEDAPDiabScreening.txt'), delimiter='|')
print(f"Screening shape: {screening.shape}")
print("\nUnique pump types:")
print(screening['PumpType'].value_counts())
print("\nCGM devices:")
print(screening['CGMUseDevice'].value_counts())

Screening shape: (105, 66)

Unique pump types:
PumpType
Tandem t:slim X2 with Basal:IQ               34
Insulet OmniPod Insulin Management System    33
Medtronic 670G                                2
Name: count, dtype: int64

CGM devices:
CGMUseDevice
Dexcom    102
Abbott      1
Name: count, dtype: int64


## Create User Data Expansion DataFrame

Following the exact same 10-column structure as DCLP3.csv:
1. id
2. insulin_delivery_device
3. insulin_delivery_algorithm
4. cgm_device
5. ethnicity
6. age_of_diagnosis
7. is_pregnant
8. insulin_delivery_modality
9. insulin_type_bolus
10. insulin_type_basal

In [4]:
# Filter completed participants only
completed_roster = roster[roster['PtStatus'] == 'Completed'].copy()
print(f"Completed participants: {len(completed_roster)}")

# Merge roster with screening data
merged_data = completed_roster.merge(screening, on='PtID', how='inner')
print(f"Merged data shape: {merged_data.shape}")

Completed participants: 96
Merged data shape: (96, 73)


In [5]:
# Create user data expansion matching DCLP3 structure exactly
user_data_expansion = pd.DataFrame()

# 1. id
user_data_expansion['id'] = merged_data['PtID']

print(f"Created user data expansion with {len(user_data_expansion)} participants")

Created user data expansion with 96 participants


In [6]:
merged_data['PumpType'].unique()

array(['Insulet OmniPod Insulin Management System', nan,
       'Tandem t:slim X2 with Basal:IQ', 'Medtronic 670G'], dtype=object)

In [7]:
# 2. insulin_delivery_device - Map pump types for PEDAP, ensuring CLC subjects get t:slim X2
def map_insulin_delivery_device(pump_type, trt_group):
    # CLC subjects MUST have t:slim X2 (study protocol requirement)
    if trt_group == 'CLC':
        return 't:slim X2'
    
    if pd.isna(pump_type):
        return np.nan
    
    pump_str = str(pump_type).strip()
    
    if 'OmniPod' in pump_str or 'Omnipod' in pump_str:
        return 'OmniPod'
    elif 'Tandem' in pump_str:
        return 't:slim X2'
    elif 'Medtronic' in pump_str:
        return pump_str
    else:
        return pump_str

user_data_expansion['insulin_delivery_device'] = merged_data.apply(
    lambda x: map_insulin_delivery_device(x['PumpType'], x['TrtGroup']), axis=1
)

print("Insulin delivery device distribution:")
print(user_data_expansion['insulin_delivery_device'].value_counts(dropna=False))

Insulin delivery device distribution:
insulin_delivery_device
t:slim X2         76
NaN               10
OmniPod            9
Medtronic 670G     1
Name: count, dtype: int64


In [8]:
# 3. insulin_delivery_algorithm - Map based on treatment group, ensuring CLC = Control-IQ
def map_insulin_delivery_algorithm(trt_group, device):
    # CLC subjects MUST have Control-IQ (study protocol requirement)
    if trt_group == 'CLC':
        return 'Control-IQ'
    elif trt_group == 'SC':  # Standard Care
        if pd.notna(device) and 't:slim X2' in str(device):
            return 'Basal-IQ'  # Predictive low glucose suspend
        elif 'OmniPod' in str(device):
            return 'basal-bolus'
        elif 'Medtronic' in str(device):
            return 'SmartGuard'
        else:
            return np.nan # Unknown, might be MDI?
    else:
        return np.nan

user_data_expansion['insulin_delivery_algorithm'] = merged_data.apply(
    lambda x: map_insulin_delivery_algorithm(x['TrtGroup'], x['PumpType']), axis=1
)

print("Insulin delivery algorithm distribution:")
print(user_data_expansion['insulin_delivery_algorithm'].value_counts())

Insulin delivery algorithm distribution:
insulin_delivery_algorithm
Control-IQ     65
Basal-IQ       11
basal-bolus     9
SmartGuard      1
Name: count, dtype: int64


In [9]:
# 4. cgm_device - PEDAP used Dexcom G6
def map_cgm_device(cgm_device):
    if pd.isna(cgm_device):
        return 'Dexcom G6'  # Default for PEDAP timeframe (2020-2021)
    elif 'Dexcom' in str(cgm_device):
        return 'Dexcom G6'  # PEDAP used G6
    else:
        return str(cgm_device)

user_data_expansion['cgm_device'] = merged_data['CGMUseDevice'].apply(map_cgm_device)

print("CGM device distribution:")
print(user_data_expansion['cgm_device'].value_counts())

CGM device distribution:
cgm_device
Dexcom G6    96
Name: count, dtype: int64


In [10]:
# 5. ethnicity - Combine ethnicity and race like DCLP3
def combine_ethnicity_race(row):
    ethnicity = str(row['Ethnicity']) if pd.notna(row['Ethnicity']) else ''
    race = str(row['Race']) if pd.notna(row['Race']) else ''
    
    if ethnicity == 'Hispanic or Latino':
        if race == 'White':
            return 'White, Hispanic/Latino'
        elif race == 'More than one race':
            return 'Hispanic/Latino, More than one race'
        else:
            return 'Hispanic/Latino'
    elif race == 'White':
        return 'White'
    elif race == 'Black/African American':
        return 'Black/African American'
    elif race == 'Asian':
        return 'Asian'
    elif race == 'More than one race':
        return 'More than one race'
    elif race == 'American Indian/Alaska Native':
        return 'American Indian/Alaska Native'
    else:
        return race if race else 'Unknown'

user_data_expansion['ethnicity'] = merged_data.apply(combine_ethnicity_race, axis=1)

print("Ethnicity distribution:")
print(user_data_expansion['ethnicity'].value_counts())

Ethnicity distribution:
ethnicity
White                                  71
White, Hispanic/Latino                  9
Black/African American                  6
More than one race                      4
Hispanic/Latino, More than one race     4
Asian                                   2
Name: count, dtype: int64


In [11]:
# 6. age_of_diagnosis - PEDAP has DiagAge (age at diagnosis)
user_data_expansion['age_of_diagnosis'] = merged_data['DiagAge'].fillna(np.nan)

print("Age of diagnosis statistics:")
print(user_data_expansion['age_of_diagnosis'].describe())
print(f"Missing age of diagnosis: {user_data_expansion['age_of_diagnosis'].isna().sum()}")

Age of diagnosis statistics:
count    96.000000
mean      1.958333
std       1.075240
min       0.000000
25%       1.000000
50%       2.000000
75%       3.000000
max       5.000000
Name: age_of_diagnosis, dtype: float64
Missing age of diagnosis: 0


In [12]:
# 7. is_pregnant - Always False for pediatric population (ages 2-5)
user_data_expansion['is_pregnant'] = False

print("Is pregnant distribution:")
print(user_data_expansion['is_pregnant'].value_counts())

Is pregnant distribution:
is_pregnant
False    96
Name: count, dtype: int64


In [13]:
# 8. insulin_delivery_modality - Map based on algorithm
def map_insulin_delivery_modality(algorithm):
    if algorithm == 'Control-IQ':
        return 'AID'  # Automated Insulin Delivery
    elif algorithm in ['Basal-IQ', 'basal-bolus', 'SmartGuard']:
        return 'SAP'  # Sensor-Augmented Pump
    else:
        return np.nan
        
user_data_expansion['insulin_delivery_modality'] = user_data_expansion['insulin_delivery_algorithm'].apply(map_insulin_delivery_modality)

print("Insulin delivery modality distribution:")
print(user_data_expansion['insulin_delivery_modality'].value_counts())

Insulin delivery modality distribution:
insulin_delivery_modality
AID    65
SAP    21
Name: count, dtype: int64


In [14]:
# 9. insulin_type_bolus - Load insulin data from PEDAPInsulin.txt
insulin_data = pd.read_csv(os.path.join(base_path, 'PEDAPInsulin.txt'), delimiter='|')
print(f"Insulin data shape: {insulin_data.shape}")
print("Insulin type start distribution:")
print(insulin_data['InsTypeStart'].value_counts())

# Process insulin data according to the logic:
# 1. Use "Started after enrollment" if available for the subject
# 2. If not available, use "In use at time of enrollment" for ANY treatment group
def get_insulin_type(pt_id, trt_group, insulin_df, insulin_category='bolus'):
    patient_insulins = insulin_df[insulin_df['PtID'] == pt_id].copy()
    
    if patient_insulins.empty:
        return np.nan
    
    # Priority 1: Started after enrollment (for any treatment group)
    started_after = patient_insulins[patient_insulins['InsTypeStart'] == 'Started after enrollment']
    
    if not started_after.empty:
        # Filter for bolus vs basal insulins
        if insulin_category == 'bolus':
            # Fast-acting insulins (bolus)
            bolus_insulins = started_after[started_after['InsulinName'].str.contains(
                'Humalog|Novolog|Fiasp|Apidra|Lispro|Aspart', case=False, na=False)]
            if not bolus_insulins.empty:
                return bolus_insulins.iloc[0]['InsulinName']
        else:  # basal
            # Long-acting insulins (basal) or pump (same as bolus)
            basal_insulins = started_after[started_after['InsulinName'].str.contains(
                'Lantus|Levemir|Tresiba|Glargine|Detemir|Degludec', case=False, na=False)]
            if not basal_insulins.empty:
                return basal_insulins.iloc[0]['InsulinName']
            # For pumps, basal = bolus insulin
            elif started_after[started_after['InsRoute'] == 'Pump'].shape[0] > 0:
                pump_insulin = started_after[started_after['InsRoute'] == 'Pump'].iloc[0]['InsulinName']
                return pump_insulin
    
    # Priority 2: In use at time of enrollment (for ANY treatment group)
    in_use = patient_insulins[patient_insulins['InsTypeStart'] == 'In use at time of enrollment']
    
    if not in_use.empty:
        if insulin_category == 'bolus':
            bolus_insulins = in_use[in_use['InsulinName'].str.contains(
                'Humalog|Novolog|Fiasp|Apidra|Lispro|Aspart', case=False, na=False)]
            if not bolus_insulins.empty:
                return bolus_insulins.iloc[0]['InsulinName']
        else:  # basal
            basal_insulins = in_use[in_use['InsulinName'].str.contains(
                'Lantus|Levemir|Tresiba|Glargine|Detemir|Degludec', case=False, na=False)]
            if not basal_insulins.empty:
                return basal_insulins.iloc[0]['InsulinName']
            # For pumps, basal = bolus insulin
            elif in_use[in_use['InsRoute'] == 'Pump'].shape[0] > 0:
                pump_insulin = in_use[in_use['InsRoute'] == 'Pump'].iloc[0]['InsulinName']
                return pump_insulin
    
    return np.nan

# Apply the insulin mapping
user_data_expansion['insulin_type_bolus'] = merged_data.apply(
    lambda x: get_insulin_type(x['PtID'], x['TrtGroup'], insulin_data, 'bolus'), axis=1
)

print("Insulin type bolus distribution:")
print(user_data_expansion['insulin_type_bolus'].value_counts(dropna=False))

Insulin data shape: (193, 14)
Insulin type start distribution:
InsTypeStart
In use at time of enrollment    141
Started after enrollment         52
Name: count, dtype: int64
Insulin type bolus distribution:
insulin_type_bolus
Humalog (Lispro)    58
Novolog (Aspart)    38
Name: count, dtype: int64


In [15]:
# 10. insulin_type_basal - Use same logic as bolus but for basal insulins
user_data_expansion['insulin_type_basal'] = merged_data.apply(
    lambda x: get_insulin_type(x['PtID'], x['TrtGroup'], insulin_data, 'basal'), axis=1
)

# For SAP and AID subjects (pump users), basal insulin should match bolus insulin
# If they have different insulins, it indicates they're actually MDI users
def fix_pump_insulin_types(row):
    modality = row['insulin_delivery_modality']
    bolus = row['insulin_type_bolus']
    basal = row['insulin_type_basal']
    
    # For pump users (SAP/AID), basal should equal bolus
    if modality in ['SAP', 'AID']:
        # If we have bolus but different basal, use bolus for basal
        if pd.notna(bolus) and pd.notna(basal) and bolus != basal:
            # Check if basal is a long-acting insulin (indicates MDI, not pump)
            if any(long_acting in str(basal).lower() for long_acting in ['lantus', 'glargine', 'levemir', 'detemir', 'tresiba', 'degludec']):
                # This suggests they're MDI users, not pump users - keep original basal
                return basal
            else:
                # True pump user - make basal match bolus
                return bolus
        elif pd.notna(bolus) and pd.isna(basal):
            # Pump user with bolus but no basal - use bolus for basal
            return bolus
        elif pd.isna(bolus) and pd.notna(basal):
            # Pump user with basal but no bolus - use basal for bolus (handle in next step)
            return basal
    
    return basal

user_data_expansion['insulin_type_basal'] = user_data_expansion.apply(fix_pump_insulin_types, axis=1)

print("Insulin type basal distribution:")
print(user_data_expansion['insulin_type_basal'].value_counts(dropna=False))

Insulin type basal distribution:
insulin_type_basal
Humalog (Lispro)                    50
Novolog (Aspart)                    34
Lantus (Glargine) 1 time per day     9
Degludec (Tresiba)                   2
Levemir (Detemir) 1 time per day     1
Name: count, dtype: int64


In [16]:
# Fix delivery modality and insulin types correctly
# Rule: If device is t:slim X2 and algorithm is Control-IQ, they are pump users (AID)
# For pump users (SAP/AID), basal insulin should match bolus insulin

def fix_delivery_modality_correct(row):
    device = row['insulin_delivery_device']
    algorithm = row['insulin_delivery_algorithm']
    modality = row['insulin_delivery_modality']
    basal = row['insulin_type_basal']
    
    # If device is t:slim X2 with Control-IQ, definitely AID (pump user)
    if device == 't:slim X2' and algorithm == 'Control-IQ':
        return 'AID'
    
    # If device is t:slim X2 with Basal-IQ, definitely SAP (pump user)
    if device == 't:slim X2' and algorithm == 'Basal-IQ':
        return 'SAP'
    
    # If device is OmniPod with Control-IQ, definitely AID (pump user)
    if device == 'OmniPod' and algorithm == 'Control-IQ':
        return 'AID'
    
    # If device is OmniPod with basal-bolus, definitely SAP (pump user)
    if device == 'OmniPod' and algorithm == 'basal-bolus':
        return 'SAP'
    
    # If device is Medtronic with SmartGuard, definitely SAP (pump user)
    if 'Medtronic' in str(device) and algorithm == 'SmartGuard':
        return 'SAP'
    
    # For cases with no device/algorithm but long-acting basal insulin, likely MDI
    if (pd.isna(device) or device == '') and (pd.isna(algorithm) or algorithm == ''):
        if pd.notna(basal) and any(long_acting in str(basal).lower() for long_acting in ['lantus', 'glargine', 'levemir', 'detemir', 'tresiba', 'degludec']):
            if '1 time per day' in str(basal) or 'once daily' in str(basal).lower():
                return 'MDI'
    
    return modality

user_data_expansion['insulin_delivery_modality'] = user_data_expansion.apply(fix_delivery_modality_correct, axis=1)

# Now fix insulin types: for pump users (SAP/AID), make basal match bolus
def fix_insulin_matching_correct(row):
    modality = row['insulin_delivery_modality']
    bolus = row['insulin_type_bolus']
    basal = row['insulin_type_basal']
    
    # For pump users (SAP/AID), basal should equal bolus
    if modality in ['SAP', 'AID']:
        if pd.notna(bolus):
            return bolus  # Make basal match bolus for pump users
        elif pd.notna(basal) and not any(long_acting in str(basal).lower() for long_acting in ['lantus', 'glargine', 'levemir', 'detemir', 'tresiba', 'degludec']):
            # If basal is not a long-acting insulin, use it for both
            return basal
        else:
            # If only long-acting basal available for pump user, return empty (we don't know pump insulin)
            return np.nan
    
    return basal  # Keep original for MDI users

user_data_expansion['insulin_type_basal'] = user_data_expansion.apply(fix_insulin_matching_correct, axis=1)

# Also fix bolus to match basal if needed for pump users
def fix_bolus_matching_correct(row):
    modality = row['insulin_delivery_modality']
    bolus = row['insulin_type_bolus']
    basal = row['insulin_type_basal']
    
    # For pump users (SAP/AID), bolus should equal basal
    if modality in ['SAP', 'AID']:
        if pd.isna(bolus) and pd.notna(basal):
            # If basal is not a long-acting insulin, use it for bolus too
            if not any(long_acting in str(basal).lower() for long_acting in ['lantus', 'glargine', 'levemir', 'detemir', 'tresiba', 'degludec']):
                return basal
    
    return bolus  # Keep original

user_data_expansion['insulin_type_bolus'] = user_data_expansion.apply(fix_bolus_matching_correct, axis=1)

print("Corrected delivery modality distribution:")
print(user_data_expansion['insulin_delivery_modality'].value_counts(dropna=False))

print("\nCorrected insulin type basal distribution:")
print(user_data_expansion['insulin_type_basal'].value_counts(dropna=False))

Corrected delivery modality distribution:
insulin_delivery_modality
AID    65
SAP    21
MDI     5
NaN     5
Name: count, dtype: int64

Corrected insulin type basal distribution:
insulin_type_basal
Humalog (Lispro)                    54
Novolog (Aspart)                    36
Lantus (Glargine) 1 time per day     4
Degludec (Tresiba)                   1
Levemir (Detemir) 1 time per day     1
Name: count, dtype: int64


## Save the Dataset

In [17]:
# Save to the same folder as other datasets
output_path = '/Users/miriamk.wolff/Documents/Repositories/Replica/egvinsulin/data/user_data_expansion/PEDAP.csv'
os.makedirs(os.path.dirname(output_path), exist_ok=True)

user_data_expansion.to_csv(output_path, index=False)
print(f"PEDAP user data expansion saved to: {output_path}")
print(f"Final dataset contains {len(user_data_expansion)} participants")

PEDAP user data expansion saved to: /Users/miriamk.wolff/Documents/Repositories/Replica/egvinsulin/data/user_data_expansion/PEDAP.csv
Final dataset contains 96 participants
